<h3>Advance Feature Engineering:</h3>
<b>Objective:</b>
<p>Convert the job posting dataset (one row per job) into a monthly time series dataset (one row per month) that can be used for forecasting.</p>


In [2]:
#importing necessary libraries
import pandas as pd
import numpy as np
#loading cleaned and engineered dataset
df=pd.read_csv("Cleaned_engineered_jobdata.csv")
df.head()

,Job_ID,Job_Title,Company,Company_Type,Industry,City,Location_Tier,Experience_Level,Job_Type,Work_Mode,...,Week,Day,Day_Name,Weekend_Posting,Salary_Category,Experience_Level_Encoded,Rating_Category,Applicant_Category,Hiring_Size,Skill_Count
0,IND2025000,Android Developer,Tech Mahindra,MNC,Information Technology,Remote,Remote,Senior (6-10 yrs),Full-Time,Remote,...,44,31,Friday,False,Premium,3,Average,High Competition,Small Hiring,3
1,IND2025001,QA Engineer,Dream11,Indian Unicorn,Information Technology,Lucknow,Tier 2,Senior (6-10 yrs),Full-Time,Hybrid,...,21,19,Monday,False,Premium,3,Average,High Competition,Small Hiring,5
2,IND2025002,Business Analyst,HAL,PSU/Govt,EdTech,Remote,Remote,Senior (6-10 yrs),Full-Time,Remote,...,34,21,Wednesday,False,High,3,Average,High Competition,Medium Hiring,3
3,IND2025003,Cybersecurity Analyst,Groww,Startup,Information Technology,Mumbai,Tier 1,Mid (3-6 yrs),Full-Time,Hybrid,...,12,18,Wednesday,False,High,2,Average,Medium Competition,Small Hiring,5
4,IND2025004,Python Developer,Oracle,MNC,EdTech,Remote,Remote,Junior (1-3 yrs),Full-Time,Remote,...,43,25,Friday,False,Medium,1,Average,Low Competition,Small Hiring,4


In [3]:
#checking for data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Job_ID                    5000 non-null   object 
 1   Job_Title                 5000 non-null   object 
 2   Company                   5000 non-null   object 
 3   Company_Type              5000 non-null   object 
 4   Industry                  5000 non-null   object 
 5   City                      5000 non-null   object 
 6   Location_Tier             5000 non-null   object 
 7   Experience_Level          5000 non-null   object 
 8   Job_Type                  5000 non-null   object 
 9   Work_Mode                 5000 non-null   object 
 10  Salary_LPA                5000 non-null   float64
 11  Skills_Required           5000 non-null   object 
 12  Education_Required        5000 non-null   object 
 13  Openings                  5000 non-null   int64  
 14  Applican

In [4]:
#As the job_posted is 'obj' convert it into 'datetime'
df['Date_Posted'] = pd.to_datetime(df['Date_Posted'])
df['Date_Posted'].dtype

dtype('<M8[ns]')

In [5]:
#Creating monthly hiring dataset
monthly_df = (df.groupby(pd.Grouper(key='Date_Posted', freq='MS')).size().reset_index(name='Job_Count'))
monthly_df

,Date_Posted,Job_Count
0,2024-06-01,215
1,2024-07-01,202
2,2024-08-01,222
3,2024-09-01,224
4,2024-10-01,239
5,2024-11-01,222
6,2024-12-01,198
7,2025-01-01,220
8,2025-02-01,192
9,2025-03-01,236


<h4>Creating lag features:</h4>
<p>Lag features is created to capture historical hiring patterns. These variables represent the number of job postings in the previous one, two, and three months. They provide historical context that enables forecasting models to learn temporal dependencies and improve prediction accuracy.</p>

In [6]:
#creating a forecasting dataset
forecasting_df = (df.groupby(pd.Grouper(key='Date_Posted', freq='MS')).size().reset_index(name='Job_Count'))
# Previous month's hiring
forecasting_df['Lag_1'] = forecasting_df['Job_Count'].shift(1)
# Two months ago
forecasting_df['Lag_2'] = forecasting_df['Job_Count'].shift(2)
# Three months ago
forecasting_df['Lag_3'] = forecasting_df['Job_Count'].shift(3)
forecasting_df.head()

,Date_Posted,Job_Count,Lag_1,Lag_2,Lag_3
0,2024-06-01,215,NaN,NaN,NaN
1,2024-07-01,202,215.0,NaN,NaN
2,2024-08-01,222,202.0,215.0,NaN
3,2024-09-01,224,222.0,202.0,215.0
4,2024-10-01,239,224.0,222.0,202.0


<h4> Creating Rolling Statistic:</h4>
<p>Rolling statistics is generated to capture smoothed hiring trends and variability over time. A 3-month and 6-month moving average is calculated to reduce short-term fluctuations, while a 3-month rolling standard deviation is computed to measure changes in hiring volatility..</p>

In [7]:
# 3-month moving average
forecasting_df['Rolling_Mean_3'] = (forecasting_df['Job_Count'].rolling(window=3).mean())
# 6-month moving average
forecasting_df['Rolling_Mean_6'] = (forecasting_df['Job_Count'].rolling(window=6).mean())
#Creating rolling standard deviation
forecasting_df['Rolling_STD_3'] = (forecasting_df['Job_Count'].rolling(window=3).std())
forecasting_df.head()

,Date_Posted,Job_Count,Lag_1,Lag_2,Lag_3,Rolling_Mean_3,Rolling_Mean_6,Rolling_STD_3
0,2024-06-01,215,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-07-01,202,215.0,NaN,NaN,NaN,NaN,NaN
2,2024-08-01,222,202.0,215.0,NaN,213.000000,NaN,10.148892
3,2024-09-01,224,222.0,202.0,215.0,216.000000,NaN,12.165525
4,2024-10-01,239,224.0,222.0,202.0,228.333333,NaN,9.291573


<h4>Creating Growth Features:</h4>
<p>This feature shows how much hiring increased or decreased compared to the previous month. It helps the forecasting model understand hiring momentum.</p>
<p>Growth Rate=(Current Month − Previous Month​)/(Previous Month)</p>
<h5>Month-over-Month Growth</h5>
<p>A Month-over-Month (MoM) Growth feature was created to measure the percentage change in job postings compared to the previous month. This feature captures hiring acceleration or decline, allowing forecasting models to recognize growth trends.</p>

In [8]:
# Month-over-Month Growth Rate
forecasting_df['MoM_Growth'] = forecasting_df['Job_Count'].pct_change()
forecasting_df[['Date_Posted', 'Job_Count', 'MoM_Growth']].head(10)

,Date_Posted,Job_Count,MoM_Growth
0,2024-06-01,215,NaN
1,2024-07-01,202,-0.060465
2,2024-08-01,222,0.099010
3,2024-09-01,224,0.009009
4,2024-10-01,239,0.066964
5,2024-11-01,222,-0.071130
6,2024-12-01,198,-0.108108
7,2025-01-01,220,0.111111
8,2025-02-01,192,-0.127273
9,2025-03-01,236,0.229167


<h4>Creating Trend Index:</h4>
<p>A sequential trend index was introduced to represent the progression of time. This numerical feature helps machine learning models identify long-term hiring trends.</p>

In [9]:
forecasting_df['Trend'] = range(1, len(forecasting_df) + 1)

<h4>Add Calender Features</h4>


In [10]:
forecasting_df['Year'] = forecasting_df['Date_Posted'].dt.year
forecasting_df['Month'] = forecasting_df['Date_Posted'].dt.month
forecasting_df['Quarter'] = forecasting_df['Date_Posted'].dt.quarter
forecasting_df.head()


,Date_Posted,Job_Count,Lag_1,Lag_2,Lag_3,Rolling_Mean_3,Rolling_Mean_6,Rolling_STD_3,MoM_Growth,Trend,Year,Month,Quarter
0,2024-06-01,215,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2024,6,2
1,2024-07-01,202,215.0,NaN,NaN,NaN,NaN,NaN,-0.060465,2,2024,7,3
2,2024-08-01,222,202.0,215.0,NaN,213.000000,NaN,10.148892,0.099010,3,2024,8,3
3,2024-09-01,224,222.0,202.0,215.0,216.000000,NaN,12.165525,0.009009,4,2024,9,3
4,2024-10-01,239,224.0,222.0,202.0,228.333333,NaN,9.291573,0.066964,5,2024,10,4


In [11]:
#Checking for null values
forecasting_df.isnull().sum()


Date_Posted       0
Job_Count         0
Lag_1             1
Lag_2             2
Lag_3             3
Rolling_Mean_3    2
Rolling_Mean_6    5
Rolling_STD_3     2
MoM_Growth        1
Trend             0
Year              0
Month             0
Quarter           0
dtype: int64

In [12]:
#droping the null values
forecast_df = forecasting_df.dropna().reset_index(drop=True)
forecast_df.head()
forecast_df.isnull().sum()

Date_Posted       0
Job_Count         0
Lag_1             0
Lag_2             0
Lag_3             0
Rolling_Mean_3    0
Rolling_Mean_6    0
Rolling_STD_3     0
MoM_Growth        0
Trend             0
Year              0
Month             0
Quarter           0
dtype: int64

In [14]:
forecast_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date_Posted     18 non-null     datetime64[ns]
 1   Job_Count       18 non-null     int64         
 2   Lag_1           18 non-null     float64       
 3   Lag_2           18 non-null     float64       
 4   Lag_3           18 non-null     float64       
 5   Rolling_Mean_3  18 non-null     float64       
 6   Rolling_Mean_6  18 non-null     float64       
 7   Rolling_STD_3   18 non-null     float64       
 8   MoM_Growth      18 non-null     float64       
 9   Trend           18 non-null     int64         
 10  Year            18 non-null     int32         
 11  Month           18 non-null     int32         
 12  Quarter         18 non-null     int32         
dtypes: datetime64[ns](1), float64(7), int32(3), int64(2)
memory usage: 1.7 KB


In [ ]:
monthly_df.to_csv("monthly_dataset.csv", index=False)
forecast_df.to_csv("forecasting_dataset.csv", index=False)